# **Автоматическое взаимное выравнивание графемных и фонемных цепочек**

Ноутбук реализует алгоритм выравнивания графем и фонем русских словоформ по фонетическому словарю, а также расчёт согласованности, энтропии, эффективности по взаимной информации и distance-based OPC.

Результат выравнивания — таблицы графемно-фонемных и фонемно-графемных соответствий с информацией о частоте, вероятности и словах, в которых эти соответствия представлены.

Результаты расчетов метрик представлены в конце ноутбука

## Содержание:

1) Подготовка: загрузка и предобработка материала

2) Сопоставление: функции последовательного сопоставления слов и транскрипций

3) Применение: применение функций к материалу и получение сопоставительных таблиц графемно-фонемных и фонемно-графемных сочетаний

4) Анализ: вычисление на основе данных таблиц метрик орфографической прозрачности

# **Подготовка**

### Загрузка зависимостей

In [3]:
%pip install gdown Levenshtein numpy pandas -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# для использования в колабе (загрузка и скачивание файлов)
from google.colab import files

In [4]:
from Levenshtein import distance

In [5]:
import numpy as np
import pandas as pd

In [6]:
import csv
import string
import ast
import re
import copy

## *Загрузка файлов*

выбор материала: односложный, многосложный

### Многосложный

In [ ]:
# загрузка файла для colab

!gdown 1QRtGK8tEMLSmqXK8hGUEfEr1cbuW6E97LlPOVi7E9m4 --format csv --output my_dictionary.csv

Downloading...
From (original): https://drive.google.com/uc?id=1QRtGK8tEMLSmqXK8hGUEfEr1cbuW6E97LlPOVi7E9m4
From (redirected): https://docs.google.com/spreadsheets/d/1QRtGK8tEMLSmqXK8hGUEfEr1cbuW6E97LlPOVi7E9m4/export?format=csv
To: /content/my_dictionary.csv
854kB [00:00, 9.27MB/s]


In [7]:
# функции очистки

def phoneme_list(mystring):
  mystring = mystring.replace('~', "")
  mystring = mystring.replace('*', "")
  mystring = mystring.replace('+', "")
  mystring = mystring.replace('/', "")
  mystring = mystring.replace('^', "")
  mystring = mystring.replace('-', "")
  mystring = mystring.replace('=', "")
  mystring = re.sub(r'\d+', '', mystring)
  mystring = mystring.split()
  return mystring

def grapheme_list(mystring):
  mystring = mystring.replace('-', "")
  mystring = list(mystring)
  return mystring

In [9]:
# создание и очистка df

df = pd.read_csv('my_dictionary.csv',
                 sep=',',
                 engine='python',
                 escapechar='\\')

df = df.map(lambda x: x.strip('"') if isinstance(x, str) else x)
df = df.rename(columns={'Слово': "word", "Орфоэпическая транскрипция": "transcript", "Количество словоупотреблений": "freq"})

df = df[["word", "transcript", "freq"]]

df['word'] = df['word'].apply(grapheme_list)
df['transcript'] = df['transcript'].apply(phoneme_list)

crutch = df.index[df['word'].apply(lambda x: x == ["ш", "е", "с", "т", "ь", "д", "е", "с", "я", "т"])][0]
df.at[crutch, 'transcript'] = ["sh", "y", "z'", "-", "-", "d'", "i", "s'", "a", "t"]

### Односложный

In [ ]:
# загрузка файла для colab

!gdown 1puoCZTGEe45GIOpIC-sQbdJoPriF82ZX5v-9RT-EDpM --format csv --output "mono_trans.csv"

Downloading...
From (original): https://drive.google.com/uc?id=1puoCZTGEe45GIOpIC-sQbdJoPriF82ZX5v-9RT-EDpM
From (redirected): https://docs.google.com/spreadsheets/d/1puoCZTGEe45GIOpIC-sQbdJoPriF82ZX5v-9RT-EDpM/export?format=csv
To: /content/mono_trans.csv
29.1kB [00:00, 27.5MB/s]


In [ ]:
# функции очистки

def grapheme_list(mystring):
  mystring = re.sub(r'\d+', '', mystring)
  mystring = list(mystring)
  return mystring

def phoneme_list(mystring):
  mystring = re.sub(r'\d+', '', mystring)
  mystring = mystring.split()
  return mystring

In [ ]:
# создание и очистка df

df = pd.read_csv("mono_trans.csv")
df = df[["word", "transcript", "freq"]]
df['word'] = df['word'].apply(grapheme_list)
df['transcript'] = df['transcript'].apply(phoneme_list)

## *Предпросмотр и сохранение*

In [9]:
# посмотреть таблицу
df.head()

,word,transcript,freq
0,"[а, б, с, о, л, ю, т, н, о]","[a, p, s, a, l', u, t, n, a]",38
1,"[а, б, х, а, з, е, ц]","[a, p, h, a, z', i, c]",6
2,"[а, в, г, у, с, т]","[a, v, g, u, s, t]",3
3,"[а, в, г, у, с, т, е]","[a, v, g, u, s', t', i]",11
4,"[а, в, т, о, б, и, о, г, р, а, ф, и, и]","[a, f, t, a, b', i, a, g, r, a, f', i, i]",3


In [41]:
# сохранить копию
cleaned_corpus = df.copy()

In [10]:
# сохранить в файл (при желании)
cleaned_corpus.to_csv('cleaned_corpus.csv', index=False)

In [ ]:
# скачать (при желании) -- здесь и далее использовать files.download в colab
files.download('cleaned_corpus.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Сопоставление**

### Константы

In [548]:
ru_alphabet = "абвгдеёжзийклмнопрстуфхцчшщъыьэюя"

# паттерны, которые вызывают разницу в длине цепочек
difficult = {
    'ае', 'аё', 'аю', 'ая', 'ее', 'её', 'ею', 'ея', 'ие', 'иё',
    'ию', 'ия', 'ое', 'оё', 'ою', 'оя', 'уе', 'уё', 'ую', 'уя',
    'ые', 'ыё', 'ыю', 'ыя', 'эе', 'эё', 'эю', 'эя', 'юе', 'юё',
    'юю', 'юя', 'ёе', 'ёё', 'ёю', 'ёя', 'яе', 'яё', 'яю', 'яя',
    'чувств', 'здравств', 'ргск', 'нгск', 'здн', 'ндск', 'нтск',
    'рдц', 'йст', 'лнц', 'стск', 'стк', 'стл', 'стн', 'тч', 'дч',
    'дц', 'сч', 'сщ', 'стч', 'здч', 'зч', 'жч', 'шч', 'зж', 'сж',
    'сш', 'зш', 'дт', 'td', 'тс', 'дс', 'тц', 'тьс', 'стм', 'зск',
    'ъя', 'ъе', 'ъю', 'ъё', 'ья', 'ье', 'ью', 'ьё', 'ьи', 'ьо',
}


# --------------- согласные ---------------

consonants = ("б", "в", "г", "д", "ж", "з", "й", "к", "л", "м", "н", "п", "р", "с", "т", "ф", "х", "ц", "ч", "ш", "щ")
consonant_sounds = (
    'b', 'p', "b'", "p'", 'v', 'f', "v'", "f'", 'g', 'k', "g'", "k'", 'h',
    'd', 't', "d'", "t'", 'z', 's', "z'", "s'", 'k', "k'", 'g', "g'",
    'p', "p'", 'b', "b'", 's', "s'", 'z', "z'", 't', "t'", 'd', "d'", 'c',
    'f', "f'", 'v', "v'", 'zh', 'sh', 'sh', 'zh', 'j', 'ch', 'c', 'sc',
    'x', "x'", 'l', "l'", 'm', "m'", 'n', "n'", 'r', "r'"
)

# допустимые пары
all_pairs = {
    "б": ("b", "p", "b'", "p'"),
    "в": ("v", "f", "v'", "f'"),
    "г": ("g", "k", "g'", "k'", "h", "v"),
    "д": ("d", "t", "d'", "t'", "c", "ch"),
    "з": ("z", "s", "z'", "s'", "zh", "sh", "sc"),
    "к": ("k", "k'", "g", "g'"),
    "п": ("p", "p'", "b", "b'"),
    "с": ("s", "s'", "z", "z'", "sh", "c", "sc", "zh"),
    "т": ("t", "t'", "d", "d'", "c", "ch"),
    "ф": ("f", "f'", "v", "v'"),
    "ж": ("zh", "sh", "sc"),
    "ш": ("sh", "zh", "sc"),
    "й": ("j"),
    "ч": ("ch", "sh", "sc"),
    "ц": ("c"),
    "щ": ("sc"),
    "х": ("h", "h'"),
    "л": ("l", "l'"),
    "м": ("m", "m'"),
    "н": ("n", "n'"),
    "р": ("r", "r'"),
}

# фонемы, которые могут появиться на месте сочетаний типа Cь
pal_sounds = (
    "zh", "sh", "b'", "p'", "v'", "f'", "g'", "k'", "d'", "t'",
    "z'", "s'", "h'", "l'", "m'", "m", "n'", "r'", "c"
)


# --------------- гласные ---------------

vowels = {"у", "о", "а", "э", "ы", "ю", "ё", "я", "е", "и"}
vowels_jot = {"е", "ё", "ю", "я"}
vowel_sounds = {"a", "e", "u", "y", "o", "i"}


## *Функция обработки простых случаев*

* длина word и transcript равна

* отсутствуют триггеры несовпадения (мягкий и твердый знаки, кластеры, зияния и пр.)

In [549]:
def same_len(word, transcript) -> list:
  """
  Проверяет слова и транскрипции на триггеры несовпадения цепочек
  Обрабатывает случаи, прошедшие проверку
  Возвращает списки пар графема-фонема / пустые списки

  """

  def valid_pairs() -> bool:
    """
    Проверка допустимости пар
    """
    for letter, sound in zip(word, transcript):
      if letter in all_pairs and sound not in all_pairs[letter]:
        return False
      else:
        return True

  forbidden_chars = {'ь', 'ъ'}
  forbidden_patterns = difficult
  word_str = "".join(word)

  conditions = [
    len(word) == len(transcript),
    not any(char in word_str for char in forbidden_chars),
    not any(pattern in word_str for pattern in forbidden_patterns),
    word[0] not in {'я', 'е', 'ю', 'ё'},
    valid_pairs()
  ]

  return list(zip(word, transcript)) if all(conditions) else []

## *Функции обработки сложных случаев*

### Функции кластеров

#### 1) Крупные кластеры

In [550]:
# чувств
def chuvstv(transcript, word, i):
  word_string = "".join(word)
  if "чувств" in word_string:
    if word[0] == 'ч':
      transcript.insert(2, '-')
    elif "счувств" in word_string and "sc" in transcript:
      transcript.insert(transcript.index('sc') + 2, '-')
    else:
      transcript.insert(transcript.index('ch') + 2, '-')

# здравств
def zdravstv(transcript, word, i):
  word_string = "".join(word)
  if "здравств" in word_string:
    if "".join(word)[:1] == 'зд':
          transcript.insert(4, '-')
    else:
          transcript.insert(word_string.index('з') + 4, '-')

# ------------------------------

# ргск
def rgsk(transcript, word, i):
  if i in "".join(word):
    if 'rsk' in "".join(transcript):
      for j in range(len(transcript) - 2):
        if transcript[j] == 'r' and transcript[j + 1] == 's' and transcript[j + 2][0] == "k":
          transcript.insert(j + 1, '-')
          break

# нгск
def ngsk(transcript, word, i):
  if i in "".join(word):
    if 'nsk' in "".join(transcript):
      for j in range(len(transcript) - 2):
        if transcript[j] == 'n' and transcript[j + 1] == 's' and transcript[j + 2][0] == "k":
          transcript.insert(j + 1, '-')
          break

# ндск
def ndsk(transcript, word, i):
  if i in "".join(word):
    if 'nsk' in "".join(transcript):
      for j in range(len(transcript) - 2):
        if transcript[j] == 'n' and transcript[j + 1] == 's' and transcript[j + 2][0] == "k":
          transcript.insert(j + 1, '-')
          break


# ------------------------------

# здн
def zdn(transcript, word, i):
  if "бездн" not in "".join(word):
    if 'zn' in "".join(transcript) or "z'n" in "".join(transcript):
      for j in range(len(transcript) - 1):
        if transcript[j][0] == "z" and transcript[j + 1][0] == "n":
          transcript.insert(j + 1, '-')
          break

# рдц
def rdc(transcript, word, i):
  if 'rc' in "".join(transcript):
    for j in range(len(transcript) - 1):
      if transcript[j] == 'r' and transcript[j + 1] == 'c':
        transcript.insert(j + 1, '-')
        break

# ------------------------------

# йст
def jst(transcript, word, i):
  if i in "".join(word) and 'jst' not in "".join(transcript):
    for j in range(len(transcript) - 1):
      if transcript[j] == 's' and transcript[j + 1] == 't':
        transcript.insert(j, '-')
        break

# лнц
def lnc(transcript, word, i):
  if 'nc' in "".join(transcript):
    for j in range(len(transcript) - 1):
      if transcript[j] == 'n' and transcript[j + 1] == 'c':
        transcript.insert(j, '-')
        break

# ------------------------------

# стск
def stsk(transcript, word, i):
  if i in "".join(word):
    if 'ssk' in "".join(transcript):
      for j in range(len(transcript) - 2):
        if transcript[j] == 's' and transcript[j + 1] == 's' and transcript[j + 2] in ("k", "k'"):
          transcript.insert(j + 1, '-')
          break

# стк
def stk(transcript, word, i):
  if i in "".join(word):
    if 'sk' in "".join(transcript):
      for j in range(len(transcript) - 2):
        if transcript[j] == 's' and transcript[j + 1] in ("k", "k'"):
          transcript.insert(j + 1, '-')
          break

# зск
def zsk(transcript, word, i):
  if i in "".join(word):
    if 'ssk' not in "".join(transcript):
      if 'sk' in "".join(transcript):
        for j in range(len(transcript) - 2):
          if transcript[j] == 's' and transcript[j + 1] in ("k", "k'"):
            transcript.insert(j, '-')
            break

# ------------------------------

# стм
def stm(transcript, word, i):
  if 'sm' in "".join(transcript):
    for j in range(len(transcript) - 1):
      if transcript[j][0] == "s" and transcript[j + 1][0] == "m":
        transcript.insert(j + 1, '-')
        break

# стл
def stl(transcript, word, i):
  if 'хваст' not in word and 'kost' not in word:
    if "sl" in "".join(transcript):
      for j in range(len(transcript) - 1):
        if "".join(word)[:j] not in ["бес", "вос", "взос", "дис", "ис", "мис", "нис", "нос", "рас", "рос", "чрес", "черес"]:
          if transcript[j] == 's' and transcript[j + 1][0] == "l":
            transcript.insert(j + 1, '-')
            break

# стн
def stn(transcript, word, i):
  if "sn" in "".join(transcript) or "s'n" in "".join(transcript):
    for j in range(len(transcript) - 1):
      if transcript[j] in ("s", "s'") and transcript[j + 1] in ("n", "n'"):
        transcript.insert(j + 1, '-')
        break

#### 2) Кластеры типа тС и дС

In [551]:
# тч
def tch(transcript, word, i):
  if i in "".join(word) and "ch" in "".join(transcript):
    if "tch" not in "".join(transcript) and "chch" not in "".join(transcript):
      j = transcript.index("ch")
      transcript.insert(j, '-')

# дч
def dch(transcript, word, i):
  if i in "".join(word) and "ch" in "".join(transcript) and "здч" not in "".join(word):
    if "dch" not in "".join(transcript) and "chch" not in "".join(transcript):
      j = transcript.index("ch")
      transcript.insert(j, '-')

# дц
def dc(transcript, word, i):
  if 'рдц' not in "".join(word):
    if i in "".join(word):
      if "cc" not in "".join(transcript) and "dc" not in "".join(transcript):
        j = transcript.index("c")
        transcript.insert(j, '-')

# дт
def dt(transcript, word, i):
  if i in "".join(word):
    if ('tt' not in "".join(transcript)) and ("t't'" not in "".join(transcript)):
      myword = word.copy()
      for j, letter in enumerate(myword[:-1]):
        if myword[j] == "д" and myword[j + 1] == "т":
          myword[j] = "".join((myword[j], myword.pop(j + 1)))

      for i, sound in enumerate(transcript[:-1]):
          if i == myword.index("дт"):
            transcript.insert(i, '-')

# тд
def td(transcript, word, i):
  if i in "".join(word):
    if ('dd' not in "".join(transcript)) and ("d'd'" not in "".join(transcript)):
      myword = word.copy()
      for j, letter in enumerate(myword[:-1]):
        if myword[j] == "т" and myword[j + 1] == "д":
          myword[j] = "".join((myword[j], myword.pop(j + 1)))

      for i, sound in enumerate(transcript[:-1]):
          if i == myword.index("тд"):
            transcript.insert(i, '-')

# ------------------------------

# тс дс тц
def tc(transcript, word, i):
  if i in "".join(word):
    k = 0
    for idx, sound in enumerate(transcript[:-1]):
      real_idx = idx + k
      if sound == "c" and word[real_idx] in ("т", "д"):
        # check only the local context around this specific 'c'
        prev = transcript[real_idx - 1] if real_idx > 0 else ""
        next_ = transcript[real_idx + 1] if real_idx + 1 < len(transcript) else ""
        if prev not in ("c", "t", "t'") and next_ not in ("c", "s", "s'"):
          transcript.insert(real_idx, '-')
          k += 1

# ------------------------------

# тьс
def tcc(transcript, word, i):
  if i in "".join(word):
    if "c" in "".join(transcript):
      if not any(sub in "".join(transcript) for sub in ["c-c", "c-s", "t-s", "t'-s", "t-c"]):
        k = 0
        for i, sound in enumerate(transcript[:-1]):
          i = i + k
          if sound == "c" and word[i] == "т":
            transcript.insert(i, '-')
            transcript.insert(i, '-')
            k += 2

#### 3) Шипящие

In [552]:
# сч
def sch(transcript, word, i):
  if i in "".join(word) and "sc" in "".join(transcript):
    if not any(sub in "".join(transcript) for sub in ["scch", "scsc"]):
      j = transcript.index("sc")
      transcript.insert(j, "-")

# сщ
def ssch(transcript, word, i):
  if i in "".join(word) and "sc" in "".join(transcript):
    if not any(sub in "".join(transcript) for sub in ["scch", "scsc"]):
      j = transcript.index("sc")
      transcript.insert(j, "-")

#шч
def shch(transcript, word, i):
  if i in "".join(word) and "sc" in "".join(transcript):
    if not any(sub in "".join(transcript) for sub in ["scch", "scsc"]):
      j = transcript.index("sc")
      transcript.insert(j, "-")

#зч
def zch(transcript, word, i):
  if i in "".join(word) and "sc" in "".join(transcript):
    if not any(sub in "".join(transcript) for sub in ["scch", "scsc"]):
      j = transcript.index("sc")
      transcript.insert(j, "-")

#жч
def zhch(transcript, word, i):
  if i in "".join(word) and "sc" in "".join(transcript):
    if not any(sub in "".join(transcript) for sub in ["scch", "scsc"]):
      j = transcript.index("sc")
      transcript.insert(j, "-")


#стч
def stch(transcript, word, i):
  if i in "".join(word) and "sc" in "".join(transcript):
    if not any(sub in "".join(transcript) for sub in ["scch", "scsc"]):
      j = transcript.index("sc")
      transcript.insert(j, "-")
      transcript.insert(j, "-")

#здч
def zdch(transcript, word, i):
  if i in "".join(word) and "sc" in "".join(transcript):
    if not any(sub in "".join(transcript) for sub in ["scch", "scsc"]):
      j = transcript.index("sc")
      transcript.insert(j, "-")
      transcript.insert(j, "-")

In [553]:
#зж
def zzh(transcript, word, i):
  if i in "".join(word) and "zh" in "".join(transcript):
    if "zh'zh'" not in "".join(transcript) and "zhzh" not in "".join(transcript):
      j = transcript.index("zh")
      transcript.insert(j, '-')

#сж
def szh(transcript, word, i):
  if i in "".join(word) and "zh" in "".join(transcript):
    if "zh'zh'" not in "".join(transcript) and "zhzh" not in "".join(transcript):
      j = transcript.index("zh")
      transcript.insert(j, '-')


#сш
def ssh(transcript, word, i):
  if i in "".join(word) and "sh" in "".join(transcript):

    if "shsh" not in "".join(transcript):
      j = transcript.index("sh")
      transcript.insert(j, '-')

### Словарь функций кластеров

In [554]:
cluster_handlers = {
      "чувств": chuvstv,
      "здравств": zdravstv,
      "ргск": rgsk,
      "нгск": ngsk,
      "здн": zdn,
      "ндск": ndsk,
      "нтск": ndsk,
      "рдц": rdc,
      "йст": jst,
      "лнц": lnc,
      "стск": stsk,
      "стк": stk,
      "стл": stl,
      "стн": stn,
      "тч": tch,
      "дч": dch,
      "дц": dc,
      "сч": sch,
      "сщ": ssch,
      "стч": stch,
      "здч": zdch,
      "зч": zch,
      "жч": zhch,
      "шч": shch,
      "зж": zzh,
      "сж": szh,
      "сш": ssh,
      "зш": ssh,
      "дт": dt,
      "тд": td,
      "тс": tc,
      "дс": tc,
      "тц": tc,
      "тьс": tcc,
      "стм": stm,
      "зск": zsk,
  }

## Прочие функции

In [555]:
# looking for ь between consonants

def soft_cc(word, transcript):
  soft_index = word.index('ь')
  if soft_index != len(word) - 1:
    if (word[soft_index - 1] in consonants) and (word[soft_index + 1] in consonants):

      # Find the ONE palatalized sound just before ь and insert '-' once
      target = word[soft_index - 1]
      for j, sound in enumerate(transcript):
        if sound in pal_sounds:
          if target in all_pairs and sound in all_pairs[target]:
            target_2 = word[soft_index + 1]
            if target_2 in all_pairs and transcript[j + 1] in all_pairs[target_2]:
              transcript.insert(j + 1, '-')
              break

In [556]:
# looking for ь between consonant and vowel

def soft_cv(word, transcript):
  jot_index = False
  for num, letter in enumerate(word):
    if letter == "ь":
      soft_index = num
      if soft_index != len(word) - 1:
        if word[soft_index + 1] in {'е', 'ё', 'ю', 'я', 'о', "и"}:
            for i, sound in enumerate(transcript[:-1]):
              if sound == "j":
                if transcript[i - 1] in all_pairs[word[soft_index - 1]] and transcript[i + 1][0] in vowel_sounds:
                  jot_index = i
                  break
            if jot_index:
              transcript[jot_index] = "".join((transcript[jot_index], transcript.pop(jot_index + 1)))
              transcript.insert(jot_index, '-')

In [557]:
#looking for jot between vowels

def jot_vv(word, transcript):
  jot_index = False
  for i in range(len(word) - 1):
    if word[i] in vowels and word[i + 1] in vowels_jot:
      if 'j' in transcript:
        for j, sound in enumerate(transcript[:-1]):
          if sound == "j":
            if transcript[j - 1][0] in vowel_sounds and transcript[j + 1][0] in vowel_sounds:
              transcript[j] = "".join((transcript[j], transcript.pop(j + 1)))
            elif len(transcript[j - 1]) > 1:
              if transcript[j - 1][1] in vowel_sounds and transcript[j + 1][0] in vowel_sounds:
                transcript[j] = "".join((transcript[j], transcript.pop(j + 1)))

In [558]:
# looking for doubles

def doubling(word, transcript):

  for num, letter in enumerate(word[:-1]):
    try:
      i = word.index(letter)
    except ValueError:
      pass

    if letter in consonants:
      if word[num + 1] != letter:
        continue
      else:
        if transcript[num] * 2 not in "".join(transcript):
          transcript.insert(num + 1, "-")


## Буква за буквой

### letter_by_letter

In [559]:

def letter_by_letter(word, transcript):

  word = word.copy()
  transcript = transcript.copy()

  word_str = "".join(word)
  tran_str = "".join(transcript)

  # -------------------------------------------
  # -------- ОБРАБОТКА СЛОЖНЫХ СЛУЧАЕВ --------

  #looking for jot at the beginning
  if word[0] in vowels_jot and transcript[0][0] == "j":
    transcript[0] = "".join((transcript[0], transcript.pop(1)))

  #looking for jot between vowels
  if 'j' in transcript:
    jot_vv(word, transcript)


  # looking for ь at the end
  if word[-1] == 'ь':
      transcript.append('-')

  # looking for ь between consonants
  if 'ь' in word:
    soft_cc(word, transcript)

  # looking for ь between consonant and vowel
  if 'ь' in word and 'j' in transcript:
    soft_cv(word, transcript)


  # looking for ъ
  if 'ъ' in word:
      transcript.insert(transcript.index('j'), '-')
      transcript[transcript.index('j')] = ''.join((transcript[transcript.index('j')], transcript.pop(transcript.index('j') + 1)))


  #looking for clusters
  word_str = "".join(word)

  for cluster, handler in cluster_handlers.items():
    while cluster in word_str:
      old = word_str
      handler(transcript, word, cluster)
      word_str = "".join(word)

      if word_str == old:
        break

  #looking for doubles
  doubling(word, transcript)


  # ---------------------------------
  # -------- ПРОХОД ПО СЛОВУ --------


  error_found = False

  for i, letter in enumerate(word):
    if i >= len(transcript):
      break
    if (letter in consonants) and (transcript[i] != "-"):
      current_sound = transcript[i]
      if letter in all_pairs.keys():
        if current_sound in all_pairs[letter]:
          pass
        else:
          error_found = True
          break

  if not error_found:
    if len(word) != len(transcript):
      corr = "error"
    else:
      corr = list(zip(word, transcript))
  else:
    corr = "error"

  return corr


## Тесты

In [560]:
print(letter_by_letter(['с', 'и', 'я', 'ю', 'т'], ["s'", 'i1', 'j', 'a0', 'j', 'u4', 't']))
print(letter_by_letter(list("отвезли"), ["v'", "i", "z", "l'", "i", "v", "b"]))
print(letter_by_letter(['п', 'р', 'о', 'н', 'и', 'к', 'н', 'о', 'в', 'е', 'н', 'н', 'о', 'с', 'т', 'ь'], ['p', 'r', 'a2', "n'", 'i1', 'k', 'n', 'a1', "v'", 'e0', 'n', "n'", 'a4', "s'", "t'"]))
print(letter_by_letter(['с', 'п', 'о', 'к', 'о', 'й', 'н', 'е', 'е'], ['s', 'p', 'a1', 'k', 'o0', 'j', "n'", 'i4', 'j', 'i4']))
print(letter_by_letter(['ч', 'е', 'р', 'е', 'с', 'ч', 'у', 'р'], ['ch', 'i1', "r'", 'i1', 'sc', 'ch', 'u0']))
print(letter_by_letter(['у', 'т', 'р', 'е', 'н', 'н', 'я', 'я'], ['u0', 't', "r'", 'i4', "n'", 'a4', 'j', 'a4']))
print(letter_by_letter(['л', 'у', 'к', 'ь', 'я', 'н', 'о', 'п', 'о', 'д', 'о', 'б', 'и', 'я'], ['l', 'u1', "k'", 'j', 'a0', 'n', 'a4', 'p', 'a1', 'd', 'o0', "b'", 'i4', 'j', 'a4']))
print(letter_by_letter(list("янтарная"), list('jintarnaja')))
print(letter_by_letter(list("амбиция"), list('ambicija')))
print(letter_by_letter(list("разъяснить"), list('razjisnit')))
print(letter_by_letter(list("бесчувственный"), 'b i sc u s t v i n y j'.split()))
print(letter_by_letter(list("здравствуйте"), 'z d r a s t v u j'.split()))
print(letter_by_letter(list("звездный"), 'z v e z n y j'.split()))
print(letter_by_letter(list("счастливый"), 'sc a s l i v y j'.split()))
print(letter_by_letter(list("марьи"), "m a r' j i".split()))


[('с', "s'"), ('и', 'i1'), ('я', 'ja0'), ('ю', 'ju4'), ('т', 't')]
error
[('п', 'p'), ('р', 'r'), ('о', 'a2'), ('н', "n'"), ('и', 'i1'), ('к', 'k'), ('н', 'n'), ('о', 'a1'), ('в', "v'"), ('е', 'e0'), ('н', 'n'), ('н', "n'"), ('о', 'a4'), ('с', "s'"), ('т', "t'"), ('ь', '-')]
[('с', 's'), ('п', 'p'), ('о', 'a1'), ('к', 'k'), ('о', 'o0'), ('й', 'j'), ('н', "n'"), ('е', 'i4'), ('е', 'ji4')]
error
[('у', 'u0'), ('т', 't'), ('р', "r'"), ('е', 'i4'), ('н', "n'"), ('н', '-'), ('я', 'a4'), ('я', 'ja4')]
[('л', 'l'), ('у', 'u1'), ('к', "k'"), ('ь', '-'), ('я', 'ja0'), ('н', 'n'), ('о', 'a4'), ('п', 'p'), ('о', 'a1'), ('д', 'd'), ('о', 'o0'), ('б', "b'"), ('и', 'i4'), ('я', 'ja4')]
[('я', 'ji'), ('н', 'n'), ('т', 't'), ('а', 'a'), ('р', 'r'), ('н', 'n'), ('а', 'a'), ('я', 'ja')]
[('а', 'a'), ('м', 'm'), ('б', 'b'), ('и', 'i'), ('ц', 'c'), ('и', 'i'), ('я', 'ja')]
[('р', 'r'), ('а', 'a'), ('з', 'z'), ('ъ', '-'), ('я', 'ji'), ('с', 's'), ('н', 'n'), ('и', 'i'), ('т', 't'), ('ь', '-')]
[('б', 'b'),

# **Применение**

In [570]:
matching_result = df.copy()

Применение сопоставления простых случаев

In [571]:
matching_result['corr'] = matching_result.apply(lambda row: same_len(row['word'], row['transcript']), axis=1)

In [572]:
empty_count = len(matching_result[matching_result['corr'].apply(len) == 0])
print(f"Rows with empty corr: {empty_count}")

Rows with empty corr: 2839


Применение сопоставления трудных случаев

In [573]:
mask = matching_result['corr'].apply(len) == 0

matching_result.loc[mask, 'corr'] = matching_result[mask].apply(
    lambda row: letter_by_letter(row['word'], row['transcript']),
    axis=1
)

In [574]:
error_count = len(matching_result[matching_result['corr'] == "error"])
print(f"Rows with error in corr: {error_count}")

Rows with error in corr: 29


Просмотр примеров ошибок

In [575]:
matching_result[matching_result['corr'] == "error"].head()

,word,transcript,freq,corr
367,"[в, а, н, е, й]","[v, a, n', i]",3,error
402,"[в, в, и, н, т, и, л]","[v, v', i, n', t', i]",2,error
458,"[в, е, с, н, у, ш, ч, а, т, о, й]","[v', i, s, n, u, sc, ch, i, t, a]",4,error
1352,"[д, о, ж, д, у, т, с, я]","[d, a, zh, d, u, c, c]",8,error
2200,"[к, б]","[k, a, b, e]",3,error


In [576]:
# очистка от ошибок
matching_result = matching_result[matching_result['corr'] != "error"]

In [577]:
matching_result.to_csv('corr.csv')

In [578]:
files.download('corr.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Создание таблиц**



## Таблица 1

*Слово -- позиция -- буква -- фонема -- частот*а

In [579]:
table_1 = pd.DataFrame(
    columns = matching_result.columns, data = copy.deepcopy(matching_result.values)
)

In [580]:
rows = []
for _, r in table_1.iterrows():
    corr_pairs = r['corr']
    if not isinstance(corr_pairs, list) or len(corr_pairs) == 0:
        continue

    word_str = ''.join(r['word']) if isinstance(r['word'], list) else str(r['word'])
    freq = int(r['freq']) if pd.notnull(r['freq']) else 0

    for pos, (gr, ph) in enumerate(corr_pairs, start=1):
        rows.append({
            'word': word_str,
            'pos': pos,
            'grapheme': gr,
            'phoneme_real': ph,
            'freq': freq,
        })

long_df = pd.DataFrame(rows)

In [581]:
# размер таблицы и предпросмотр
print('long_df shape:', long_df.shape)
long_df.head(3)

long_df shape: (55603, 5)


,word,pos,grapheme,phoneme_real,freq
0,абсолютно,1,а,a,38
1,абсолютно,2,б,p,38
2,абсолютно,3,с,s,38


In [582]:
# сохранить в файл
long_df.to_csv(
    'phoneme_realizations_by_position.csv',
    index=False
)

In [583]:
# скачать файл
files.download('phoneme_realizations_by_position.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



---


## Таблица 2

*графема - фонема - абс. - число словоформ - относительная частота (от всех пар для всех букв)*

In [584]:
try:
  long_df = long_df.copy()
except NameError:
  long_df = pd.read_csv("phoneme_realizations_by_position.csv")

In [585]:
gp_freq = (
    long_df
    .groupby(['grapheme', 'phoneme_real'], as_index=False)['freq']
    .sum()
    .rename(columns={'freq': 'total_freq'})
    .sort_values('total_freq', ascending=False)
    .reset_index(drop=True)
)

# число уникальных слов, где встречается пара
gp_word_count = (
    long_df
    .groupby(['grapheme', 'phoneme_real'])['word']
    .nunique()
    .reset_index()
    .rename(columns={'word': 'word_count'})
)

gp_freq = gp_freq.merge(gp_word_count, on=['grapheme', 'phoneme_real'])

# относительная частота
total = gp_freq['total_freq'].sum()
gp_freq['rel_freq'] = (gp_freq['total_freq'] / total * 100).round(4)

In [586]:
# размер таблицы и предпросмотр
print(gp_freq.shape)
gp_freq.head(3)

(109, 5)


,grapheme,phoneme_real,total_freq,word_count,rel_freq
0,а,a,55690,3616,8.4924
1,о,a,47748,3231,7.2813
2,и,i,37823,2909,5.7678


In [587]:
# сохранить в файл
gp_freq.to_csv('grapheme_phoneme_frequencies.csv', index=False)

In [588]:
# скачать файл
files.download('grapheme_phoneme_frequencies.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



---


## Таблица 3

*графема - фонема - абсолютная частота - число словоформ - относительная частота (от пар этой графемы)*

In [589]:
try:
  long_df = long_df.copy()
except NameError:
  long_df = pd.read_csv("phoneme_realizations_by_position.csv")

In [590]:
letter_order = {ch: i for i, ch in enumerate(ru_alphabet)}

gp_freq1 = (
    long_df
    .groupby(['grapheme', 'phoneme_real'], as_index=False)['freq']
    .sum()
    .rename(columns={'freq': 'total_freq'})
)

# уникальные слова для пары
gp_word_count = (
    long_df
    .groupby(['grapheme', 'phoneme_real'])['word']
    .nunique()
    .reset_index()
    .rename(columns={'word': 'word_count'})
)

gp_freq1 = gp_freq1.merge(gp_word_count, on=['grapheme', 'phoneme_real'])

# абсолютная частота для графемы
grapheme_totals = (
    gp_freq1
    .groupby('grapheme')['total_freq']
    .transform('sum')
)

# процент
gp_freq1['rel_freq_pct'] = (gp_freq1['total_freq'] / grapheme_totals * 100).round(2)

# сортировать: по алфавиту, по частоте внутри буквы
gp_freq1['_sort_key'] = gp_freq1['grapheme'].apply(lambda x: letter_order.get(x[0], 999))
gp_freq1 = (
    gp_freq1
    .sort_values(['_sort_key', 'total_freq'], ascending=[True, False])
    .drop(columns='_sort_key')
    .reset_index(drop=True)
)

In [591]:
# размеры таблицы и предпросмотр
print(gp_freq1.shape)
display(gp_freq1.head(3))

(109, 5)


,grapheme,phoneme_real,total_freq,word_count,rel_freq_pct
0,а,a,55690,3616,99.18
1,а,i,428,35,0.76
2,а,y,31,3,0.06


In [592]:
# сохранить
gp_freq1.to_csv('grapheme_phoneme_probabilities.csv', index=False)

In [593]:
# скачать
files.download('grapheme_phoneme_probabilities.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



---





---

## Таблица 4

*фонема - графема - абсолютная частота - число словоформ - относительная частота (от пар этой фонемы)*

In [594]:
try:
  gp_freq1 = gp_freq1.copy()
except NameError:
  onset_df = pd.read_csv("grapheme_phoneme_probabilities.csv")

In [595]:
try:
  long_df = long_df.copy()
except NameError:
  long_df = pd.read_csv("phoneme_realizations_by_position.csv")

In [596]:
phonemes = sorted(gp_freq1['phoneme_real'].unique())
phoneme_order = {ph: i for i, ph in enumerate(phonemes)}

# подсчет полной частоты
prob_pg = (
    long_df
    .groupby(['phoneme_real', 'grapheme'], as_index=False)['freq']
    .sum()
    .rename(columns={'freq': 'total_freq'})
)

# уникальные слова для пар
pf_word_count = (
    long_df
    .groupby(['phoneme_real', 'grapheme'])['word']
    .nunique()
    .reset_index()
    .rename(columns={'word': 'word_count'})
)

prob_pg = prob_pg.merge(pf_word_count, on=['phoneme_real', 'grapheme'])

# сумма частот для фонемы
phoneme_totals = (
    prob_pg
    .groupby('phoneme_real')['total_freq']
    .transform('sum')
)

# процент
prob_pg['rel_freq_pct'] = (prob_pg['total_freq'] / phoneme_totals * 100).round(2)

# сортировать: по алфавиту, по частоте внутри фонемы
prob_pg['_sort_key'] = prob_pg['phoneme_real'].apply(lambda x: phoneme_order.get(x, 999))
prob_pg = (
    prob_pg
    .sort_values(['_sort_key', 'total_freq'], ascending=[True, False])
    .drop(columns='_sort_key')
    .reset_index(drop=True)
)

In [597]:
# предпросмотр и размер
print(prob_pg.shape)
prob_pg.head()

(109, 5)


,phoneme_real,grapheme,total_freq,word_count,rel_freq_pct
0,-,ь,15372,1279,73.62
1,-,т,2211,255,10.59
2,-,н,1062,147,5.09
3,-,с,716,73,3.43
4,-,д,616,54,2.95


In [598]:
# сохранить
prob_pg.to_csv('phoneme_grapheme_probabilities.csv', index=False)

In [599]:
# скачать
files.download('phoneme_grapheme_probabilities.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



---



## Вспомогательная таблица: пары с примерами слов

для проверки адекватности сопоставления: отсортирована от более редких пар к более частотным

In [601]:
words_per_pair = (
    long_df.groupby(['grapheme', 'phoneme_real'])['word']
    .apply(list)
    .reset_index()
)


rare_pairs_with_words = (
    gp_freq.sort_values('total_freq')
    [['grapheme', 'phoneme_real', 'total_freq']]
    .merge(words_per_pair, on=['grapheme', 'phoneme_real'], how='left')
    .reset_index(drop=True)
)

In [602]:
rare_pairs_with_words

,grapheme,phoneme_real,total_freq,word
0,с,sc,1,[исчезнувшие]
1,з,sh,2,[замерзшая]
2,ш,-,3,[веснушчатое]
3,г,-,3,[петербургский]
4,с,zh,6,"[сжатый, сжималось]"
...,...,...,...,...
104,н,n,24417,"[абсолютно, автобусной, автоматная, автомобиль..."
105,т,t,30239,"[абсолютно, август, автобиографии, автобусе, а..."
106,и,i,37823,"[автобиографии, автобиографии, автобиографии, ..."
107,о,a,47748,"[абсолютно, абсолютно, автобиографии, автобиог..."


In [603]:
# сохранить
rare_pairs_with_words.to_csv("rp.csv")

In [604]:
# скачать
files.download('rp.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Вычисление метрик**

## *Процент слов с отношением букв и фонем 1:1*

In [605]:
# без частот (type count)

match_count = cleaned_corpus.apply(lambda r: len(r['word']) == len(r['transcript']), axis=1).sum()
print(f"Words where len(word) == len(transcript): {match_count} / {len(cleaned_corpus)} // {match_count / len(cleaned_corpus)}")

Words where len(word) == len(transcript): 5025 / 7303 // 0.6880733944954128


In [606]:
# умножая на частоты (token count)

match_count = cleaned_corpus[cleaned_corpus.apply(lambda r: len(r['word']) == len(r['transcript']), axis=1)]['freq'].sum()
print(f"Words where where len(word) == len(transcript): {match_count} / {cleaned_corpus['freq'].sum()} // {match_count / cleaned_corpus['freq'].sum()}")

Words where where len(word) == len(transcript): 81227 / 105571 // 0.7694063710678122


## *Число уникальных gpc*

In [607]:
print(f"\nTotal unique grapheme-phoneme pairs: {len(gp_freq)}")


Total unique grapheme-phoneme pairs: 109


## *Полная энтропия*

### Полная энтропия: графемы - фонемы

In [608]:
try:
  gp_entropy = gp_freq.copy()
except NameError:
  gp_entropy = pd.read_csv("grapheme_phoneme_frequencies.csv")

In [609]:
def calc_gp_entropy(group):
  """
  Вычисляет энтропию для группы произношений одной графемы.

  Параметры
  group : pd.DataFrame
      Подтаблица для одной графемы со столбцами:
      - ``total_freq``    : абсолютная частота произношения
      - ``phoneme_real``  : фонема (строка)

  Возвращает таблицу c информацией о фонемных парах и значениями частоты и энтропии для каждой графемы
  """
  total = group["total_freq"].sum()
  probs = group["total_freq"] / total
  H = -np.sum(probs * np.log2(probs.clip(lower=1e-10)))
  variants = (
    group.assign(prob=probs)
    .sort_values("prob", ascending=False)
    .apply(lambda row: f"{row['phoneme_real']} ({row['prob']*100:.1f}%)", axis=1)
    .tolist()
  )
  return pd.Series({
    "entropy_H":       round(H, 4),
    "total_freq_sum":  total,
    "n_pronunciations": len(group),
    "top_phoneme":      group.loc[group["total_freq"].idxmax(), "phoneme_real"],
    "top_phoneme_freq":  group["total_freq"].max(),
    "top_phoneme_pct":  round(group["total_freq"].max() / total * 100, 2),
    "variants":        ", ".join(variants)
  })

gp_entropy = (
  gp_entropy.groupby("grapheme")
  .apply(calc_gp_entropy)
  .reset_index()
  .sort_values("entropy_H", ascending=False)
)

/tmp/ipykernel_3107/1499835745.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_gp_entropy)


In [610]:
# таблица значений

print(gp_entropy.to_string(index=False))

grapheme  entropy_H  total_freq_sum  n_pronunciations top_phoneme  top_phoneme_freq  top_phoneme_pct                                                                                        variants
       е     1.9242           50955                 6           i             23186            45.50                                  i (45.5%), e (33.5%), y (7.5%), o (5.2%), ji (5.0%), je (3.2%)
       я     1.6018           12860                 4           a              7257            56.43                                                     a (56.4%), i (21.0%), ja (17.2%), ji (5.4%)
       г     1.4959            8957                 6           g              5888            65.74                                   g (65.7%), v (17.0%), g' (8.5%), k (7.6%), h (1.2%), - (0.0%)
       д     1.4643           22968                 7           d             13357            58.15                       d (58.2%), d' (32.5%), t (4.4%), - (2.7%), t' (1.9%), c (0.2%), ch (0.1%)
       с     1.

In [611]:
# энтропия по всему набору графем
gp_entropy["weight"] = gp_entropy["total_freq_sum"] / gp_entropy["total_freq_sum"].sum()
weighted_H = (summary["entropy_H"] * gp_entropy["weight"]).sum()
print(f"\nWeighted average entropy H: {weighted_H:.4f} bits")


Weighted average entropy H: 0.1769 bits


In [612]:
# сохранить
gp_entropy.to_csv("gp_entropy.csv")

In [613]:
# скачать
files.download("gp_entropy.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



---

### Полная энтропия: фонемы - графемы

In [614]:
try:
  pg_entropy = prob_pg.copy()
except NameError:
  pg_entropy = pd.read_csv("phoneme_grapheme_probabilities.csv")

In [615]:
# ----- разложение составных пар типа jV (йотированных) на два отдельных фонемных компонента -----
compound_map = {
  "ja": ["j", "a"],
  "je": ["j", "e"],
  "jo": ["j", "o"],
  "ju": ["j", "u"],
  "ji": ["j", "i"],
}

compound_rows = pg_entropy[pg_entropy["phoneme_real"].isin(compound_map.keys())]
normal_rows   = pg_entropy[~pg_entropy["phoneme_real"].isin(compound_map.keys())]

# дублирование строк с jV-парами
extra_rows = []
for _, row in compound_rows.iterrows():
  for ph in compound_map[row["phoneme_real"]]:
    extra_rows.append({
      "grapheme":    row["grapheme"],
      "phoneme_real": ph,
      "total_freq":  row["total_freq"],
      **{c: row[c] for c in pg_entropy.columns
          if c not in ("grapheme", "phoneme_real", "total_freq")}
      })

# объединение в одну таблицу
pg_entropy_expanded = pd.concat(
  [normal_rows, pd.DataFrame(extra_rows)],
  ignore_index=True
)

# агрегация
agg_cols = {c: "first" for c in pg_entropy_expanded.columns
            if c not in ("grapheme", "phoneme_real", "total_freq")}
agg_cols["total_freq"] = "sum"

pg_entropy = (
  pg_entropy_expanded
  .groupby(["grapheme", "phoneme_real"], as_index=False)
  .agg(agg_cols)
)

# ----- вычисление энтропии -----
def calc_pg_entropy(group) -> pd.Series:
  """
  Вычисляет энтропию для группы графических реализаций одной фонемы

  Параметры
  group : pd.DataFrame
      Подтаблица для одной графемы со столбцами:
      - ``total_freq``    : абсолютная частота произношения
      - ``grapheme``  : графема

  Возвращает таблицу фонемных пар и значений частоты и энтропии для каждой графемы
  """
  total = group["total_freq"].sum()
  probs = group["total_freq"] / total
  H = -np.sum(probs * np.log2(probs.clip(lower=1e-10)))
  variants = (
    group.assign(prob=probs)
    .sort_values("prob", ascending=False)
    .apply(lambda row: f"{row['grapheme']} ({row['prob']*100:.1f}%)", axis=1)
    .tolist()
  )
  return pd.Series({
    "entropy_H":         round(H, 4),
    "total_freq_sum":    total,
    "n_realizations":    len(group),
    "top_grapheme":      group.loc[group["total_freq"].idxmax(), "grapheme"],
    "top_grapheme_freq": group["total_freq"].max(),
    "top_grapheme_pct":  round(group["total_freq"].max() / total * 100, 2),
    "variants":        ", ".join(variants)
  })

pg_entropy = (
  pg_entropy.groupby("phoneme_real")
  .apply(calc_pg_entropy)
  .reset_index()
  .sort_values("entropy_H", ascending=False)
)

/tmp/ipykernel_3107/280547329.py:76: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_pg_entropy)


In [616]:
print("\nEntropy Summary (per phoneme):")
print(pg_entropy.to_string(index=False))


Entropy Summary (per phoneme):
phoneme_real  entropy_H  total_freq_sum  n_realizations top_grapheme  top_grapheme_freq  top_grapheme_pct                                                                                                                                                                   variants
           j     1.9857           18077               6            й               7751             42.88                                                                                                             й (42.9%), е (23.3%), я (16.0%), ю (15.6%), ё (2.0%), и (0.2%)
           -     1.5163           20879              17            ь              15372             73.62 ь (73.6%), т (10.6%), н (5.1%), с (3.4%), д (3.0%), ъ (1.6%), в (0.9%), ж (0.6%), л (0.5%), й (0.2%), п (0.1%), м (0.1%), к (0.1%), р (0.1%), з (0.1%), г (0.0%), ш (0.0%)
           y     1.3456           17617               5            ы              11095             62.98                                

In [617]:
# энтропия по всему набору фонем (считая -)
pg_entropy["weight"] = pg_entropy["total_freq_sum"] / pg_entropy["total_freq_sum"].sum()
weighted_H = (pg_entropy["entropy_H"] * pg_entropy["weight"]).sum()
print(f"\nWeighted average entropy H: {weighted_H:.4f} bits")

# энтропия по всему набору фонем (не считая -)
pg_entropy_no_dash = pg_entropy[pg_entropy["phoneme_real"] != "-"].copy()
pg_entropy_no_dash["weight"] = pg_entropy_no_dash["total_freq_sum"] / pg_entropy_no_dash["total_freq_sum"].sum()
weighted_H_no_dash = (pg_entropy_no_dash["entropy_H"] * pg_entropy_no_dash["weight"]).sum()
print(f"Weighted average entropy H (excluding '-'): {weighted_H_no_dash:.4f} bits")


Weighted average entropy H: 0.6400 bits
Weighted average entropy H (excluding '-'): 0.6116 bits


In [618]:
# сохранить
pg_entropy.to_csv("pg_entropy_dash.csv")
pg_entropy_no_dash.to_csv("pg_entropy_no_dash.csv")

In [620]:
# скачать
files.download("pg_entropy_dash.csv")
files.download("pg_entropy_no_dash.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## *Энтропия гласных (направление чтения)*

In [621]:
try:
  pos_df = long_df.copy()
except NameError:
  pos_df = pd.read_csv("phoneme_realizations_by_position.csv")

In [622]:
# оставляем гласные
VOWELS = vowels
vow_df = pos_df[pos_df["grapheme"].isin(VOWELS)].copy()
vow_df.rename(columns={'freq': 'total_freq'}, inplace=True)

### контекстно-независимая

In [623]:
ci_summary = (
  vow_df
  .groupby(["grapheme", "phoneme_real"])["total_freq"].sum()
  .reset_index()
  .groupby("grapheme")
  .apply(calc_gp_entropy)
  .reset_index()
  .sort_values("entropy_H", ascending=False)
)

/tmp/ipykernel_3107/2850206619.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_gp_entropy)


In [624]:
# по буквам
print(ci_summary.to_string(index=False))

grapheme  entropy_H  total_freq_sum  n_pronunciations top_phoneme  top_phoneme_freq  top_phoneme_pct                                                       variants
       е     1.9242           50955                 6           i             23186            45.50 i (45.5%), e (33.5%), y (7.5%), o (5.2%), ji (5.0%), je (3.2%)
       я     1.6018           12860                 4           a              7257            56.43                    a (56.4%), i (21.0%), ja (17.2%), ji (5.4%)
       ё     0.9992             706                 2          jo               365            51.70                                          jo (51.7%), o (48.3%)
       ю     0.9821            4875                 2          ju              2821            57.87                                          ju (57.9%), u (42.1%)
       о     0.9229           72125                 2           a             47748            66.20                                           a (66.2%), o (33.8%)
       э     0.4

In [625]:
# среднее
ci_summary["weight"] = ci_summary["total_freq_sum"] / ci_summary["total_freq_sum"].sum()
weighted_H_ci = (ci_summary["entropy_H"] * ci_summary["weight"]).sum()
print(f"\nWeighted average H (context-independent): {weighted_H_ci:.4f} bits")


Weighted average H (context-independent): 0.7797 bits


### контекстно-зависимая

In [626]:
def get_neighbor(pos_df, word, pos, offset) -> str | None:
  """
  Возвращает графему соседней позиции в слове.

  Параметры
  ----------
  pos_df : pd.DataFrame
      Таблица с позиционной разметкой слова, содержащая столбцы:
      - ``word``     : слово (строка)
      - ``pos``      : позиция графемы в слове (int)
      - ``grapheme`` : графема на данной позиции (строка)
  word : str
      Слово, в котором выполняется поиск соседа.
  pos : int
      Текущая позиция графемы.
  offset : int
      Смещение относительно текущей позиции (например, -1 — предыдущая, +1 — следующая).

  Возвращает
  ----------
  str или None
      Графему на позиции ``pos + offset``, или ``None``, если такая позиция отсутствует.
  """
  neighbor = pos_df[(pos_df["word"] == word) & (pos_df["pos"] == pos + offset)]
  if neighbor.empty:
    return None
  return neighbor.iloc[0]["grapheme"]

# lookup: (word, pos) -> grapheme  для вытаскивания соседей
lookup = pos_df.set_index(["word", "pos"])["grapheme"].to_dict()

def neighbor_grapheme(word, pos, offset):
  return lookup.get((word, pos + offset), None)

# добавляем соседей
vow_df = vow_df.copy()
vow_df["onset"] = vow_df.apply(
  lambda r: neighbor_grapheme(r["word"], r["pos"], -1), axis=1
)
vow_df["coda"] = vow_df.apply(
  lambda r: neighbor_grapheme(r["word"], r["pos"], +1), axis=1
)

In [627]:
def calc_conditional_entropy(vow_df, context_col, context_label):
  """
  Вычисляет условную энтропию для графем гласных в заданном контексте.

  Для каждой графемы энтропия взвешивается по частоте контекстных ячеек,
  что даёт H(фонема | графема, контекст). Выводит результаты в консоль
  и возвращает таблицу с энтропией по графемам.

  Параметры
  ----------
  vow_df : pd.DataFrame
      Таблица гласных с частотами произношений, содержащая столбцы:
      - ``grapheme``    : графема (строка)
      - ``phoneme_real`` : реализованная фонема (строка)
      - ``freq``        : частота данной реализации (int/float)
      - ``context_col`` : контекстный признак (см. ниже)
  context_col : str
      Название столбца с контекстным признаком (например, ``"prev_grapheme"``
      или ``"next_grapheme"``).
  context_label : str
      Человекочитаемое название контекста для вывода в консоль
      (например, ``"левый сосед"``).

  Возвращает
  ----------
  pd.DataFrame
      Таблица с одной строкой на графему и столбцами:
      - ``grapheme``      : графема
      - ``entropy_H``     : условная энтропия (бит), округлённая до 4 знаков
      - ``total_freq_sum`` : суммарная частота графемы в данном контексте
      - ``n_contexts``    : количество уникальных контекстных значений
      - ``weight``        : доля графемы в общей частоте выборки
  """
  ctx_df = vow_df.dropna(subset=[context_col]).copy()

  # cуммирование частот по тройкам (графема, контекст, фонема)
  grouped = (
    ctx_df
    .groupby(["grapheme", context_col, "phoneme_real"])["freq"]
    .sum()
    .reset_index()
  )

  # энтропия внутри одной контекстной ячейки (графема + конкретный контекст)
  def cell_entropy(g):
    """
    Вычисляет энтропию и суммарную частоту для одной контекстной ячейки.

    Параметры
    ----------
    g : pd.DataFrame
        Строки одной группы (grapheme, context_col) с столбцом ``freq``.

    Возвращает
    ----------
    pd.Series с полями ``H`` (энтропия, бит) и ``cell_freq`` (сумма частот).
    """
    total = g["freq"].sum()
    probs = g["freq"] / total
    H = -np.sum(probs * np.log2(probs.clip(lower=1e-10)))
    return pd.Series({"H": H, "cell_freq": total})

  # применение cell_entropy к каждой паре (графема, контекст)
  cell_H = (
    grouped
    .groupby(["grapheme", context_col])
    .apply(cell_entropy)
    .reset_index()
  )

  # взвешивание энтропии ячеек по доле контекста внутри графемы
  grapheme_totals = cell_H.groupby("grapheme")["cell_freq"].sum().rename("grapheme_total")
  cell_H = cell_H.join(grapheme_totals, on="grapheme")
  cell_H["weight"] = cell_H["cell_freq"] / cell_H["grapheme_total"]
  cell_H["weighted_H"] = cell_H["H"] * cell_H["weight"]

  # агрегация взвешенных энтропий до уровня графемы
  per_grapheme = (
    cell_H
    .groupby("grapheme")
    .agg(
      entropy_H=("weighted_H", "sum"),
      total_freq_sum=("cell_freq", "sum"),
      n_contexts=(context_col, "count"),
    )
    .reset_index()
  )

  # округление итоговых значений энтропии
  per_grapheme["entropy_H"] = per_grapheme["entropy_H"].round(4)

  print(f"тип контекста — {context_label}")
  print(per_grapheme.sort_values("entropy_H", ascending=False).to_string(index=False))

  # вычисление средневзвешенной энтропии по всем графемам и вывод итога
  per_grapheme["weight"] = (
      per_grapheme["total_freq_sum"] / per_grapheme["total_freq_sum"].sum()
  )
  weighted_H = (per_grapheme["entropy_H"] * per_grapheme["weight"]).sum()
  print(f"\nWeighted average H ({context_label}): {weighted_H:.4f} bits")

  return per_grapheme

In [630]:
vow_df.rename(columns={'total_freq': 'freq'}, inplace=True)
onset_summary = calc_conditional_entropy(vow_df, "onset", "conditional on onset")

тип контекста — conditional on onset
grapheme  entropy_H  total_freq_sum  n_contexts
       е     1.1598         49704.0          29
       о     0.8643         64676.0          24
       я     0.8512         12604.0          21
       э     0.4062           145.0           5
       а     0.0314         55542.0          24
       и     0.0000         37227.0          28
       ы     0.0000         11095.0          13
       у     0.0000         15180.0          25
       ю     0.0000          4833.0          15
       ё     0.0000           706.0           7

Weighted average H (conditional on onset): 0.5009 bits


/tmp/ipykernel_3107/1816049982.py:67: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(cell_entropy)


In [631]:
coda_summary  = calc_conditional_entropy(vow_df, "coda",  "conditional on coda")

тип контекста — conditional on coda
grapheme  entropy_H  total_freq_sum  n_contexts
       е     1.5881         42451.0          29
       я     1.2749          6336.0          24
       о     0.8964         60042.0          30
       и     0.3556         33361.0          30
       э     0.2230          2802.0          10
       ю     0.1914          2900.0          18
       ё     0.0785           700.0           5
       а     0.0553         46859.0          28
       ы     0.0000          8923.0          23
       у     0.0000         14268.0          27

Weighted average H (conditional on coda): 0.6632 bits


/tmp/ipykernel_3107/1816049982.py:67: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(cell_entropy)


## *Начальная энтропия*

### Пары для первых букв

In [632]:
try:
  onset_df = mydata.copy()
except NameError:
  onset_df = pd.read_csv("corr.csv")


In [633]:
onset_df = onset_df[["freq", "corr"]]
onset_df = onset_df[onset_df["corr"] != "error"]
onset_df = onset_df.dropna(subset=["corr"])
onset_df["corr"] = onset_df["corr"].apply(lambda x: eval(x) if isinstance(x, str) else x)
onset_df["corr"] = onset_df["corr"].apply(lambda x: x[0])

In [634]:
# Направление графема → фонема
gp_onset_df = onset_df.copy()

# разделяем corr на две колонки
"""
Чтобы выбрать, что сопоставлять йотированным буквам:
сочетания jV или только j
нужно заккоментировать одну из строк ниже
"""

# gp_onset_df[['grapheme', 'phoneme_real']] = onset_df['corr'].apply(lambda x: pd.Series([x[0], x[1]])) # полностью сочетания с йотом
gp_onset_df[['grapheme', 'phoneme_real']] = gp_onset_df['corr'].apply(lambda x: pd.Series([x[0], x[1][0]] if x[1].startswith('j') else [x[0], x[1]])) # только йот

# убираем нули (по идее их и не должно быть)
gp_onset_df = gp_onset_df[gp_onset_df['phoneme_real'] != '-']

# группировка и расчет
gp_onset_df = (gp_onset_df.groupby(['grapheme', 'phoneme_real'])
          .agg(total_freq=('freq', 'sum'),
               word_count=('freq', 'count'))
          .reset_index())

# добавляем процент
gp_onset_df['rel_freq_pct'] = gp_onset_df.groupby('grapheme')['total_freq'].transform(
    lambda x: (x / x.sum() * 100).round(2)
)

# сортируем
gp_onset_df = gp_onset_df.sort_values(['grapheme', 'rel_freq_pct'], ascending=[True, False])

In [635]:
gp_onset_df.head()

,grapheme,phoneme_real,total_freq,word_count,rel_freq_pct
0,а,a,607,71,100.00
1,б,b,3971,200,87.47
2,б,b',569,81,12.53
4,в,v,4457,368,58.08
5,в,v',1837,104,23.94


In [636]:
# Направление фонема → графема
pg_onset_df = onset_df.copy()

# разделяем corr на две колонки
pg_onset_df[['grapheme', 'phoneme_real']] = pg_onset_df['corr'].apply(lambda x: pd.Series([x[0], x[1]]))
pg_onset_df = pg_onset_df[pg_onset_df['phoneme_real'] != '-']

pg_onset_df = (pg_onset_df.groupby(['phoneme_real', 'grapheme'])
          .agg(total_freq=('freq', 'sum'),
               word_count=('freq', 'count'))
          .reset_index())

# добавляем процент
pg_onset_df['rel_freq_pct'] = pg_onset_df.groupby('phoneme_real')['total_freq'].transform(
    lambda x: (x / x.sum() * 100).round(2)
)

# сортируем
pg_onset_df = pg_onset_df.sort_values(['phoneme_real', 'rel_freq_pct'], ascending=[True, False])

In [637]:
pg_onset_df.head()

,phoneme_real,grapheme,total_freq,word_count,rel_freq_pct
1,a,о,4387,421,87.85
0,a,а,607,71,12.15
2,b,б,3971,200,100.00
3,b',б,569,81,100.00
4,c,ц,273,33,100.00


### Выполнение расчета

#### *с учетом частот -- для графем*

In [638]:
freq_gp_onset_df = gp_onset_df.copy()

In [639]:
summary = (
    freq_gp_onset_df.groupby("grapheme")
    .apply(calc_gp_entropy)
    .reset_index()
    .sort_values("entropy_H", ascending=False)
)

/tmp/ipykernel_3107/3339567793.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_gp_entropy)


In [640]:
print("\nEntropy Summary:")
print(summary.to_string(index=False))


Entropy Summary:
grapheme  entropy_H  total_freq_sum  n_pronunciations top_phoneme  top_phoneme_freq  top_phoneme_pct                                              variants
       в     1.3942            7674                 3           v              4457            58.08                      v (58.1%), v' (23.9%), f (18.0%)
       ф     0.9999             564                 2           f               286            50.71                                 f (50.7%), f' (49.3%)
       н     0.9878            6498                 2           n              3671            56.49                                 n (56.5%), n' (43.5%)
       о     0.9771            7449                 2           a              4387            58.89                                  a (58.9%), o (41.1%)
       ч     0.9402            2448                 2          ch              1574            64.30                                ch (64.3%), sh (35.7%)
       д     0.8299            7417                 

In [641]:
summary["weight"] = summary["total_freq_sum"] / summary["total_freq_sum"].sum()
weighted_H = (summary["entropy_H"] * summary["weight"]).sum()
print(f"\nOnset entropy tokens H: {weighted_H:.4f} bits")


Onset entropy tokens H: 0.6557 bits


#### *с учетом частот -- для фонем*

In [642]:
freq_pg_onset_df = pg_onset_df.copy()

In [643]:
summary = (
    freq_pg_onset_df.groupby("phoneme_real")
    .apply(calc_pg_entropy)
    .reset_index()
    .sort_values("entropy_H", ascending=False)
)

/tmp/ipykernel_3107/3584035431.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_pg_entropy)


In [644]:
print("\nEntropy Summary:")
print(summary.to_string(index=False))


Entropy Summary:
phoneme_real  entropy_H  total_freq_sum  n_realizations top_grapheme  top_grapheme_freq  top_grapheme_pct             variants
          z'     0.9997             304               2            с                155             50.99 с (51.0%), з (49.0%)
          sh     0.9983            1666               2            ч                874             52.46 ч (52.5%), ш (47.5%)
          ji     0.9369             218               2            е                141             64.68 е (64.7%), я (35.3%)
           y     0.8234             128               2            и                 95             74.22 и (74.2%), э (25.8%)
           f     0.6615            1666               2            в               1380             82.83 в (82.8%), ф (17.2%)
           a     0.5338            4994               2            о               4387             87.85 о (87.8%), а (12.2%)
           z     0.2903            3614               2            з               3430      

In [645]:
summary["weight"] = summary["total_freq_sum"] / summary["total_freq_sum"].sum()
weighted_H = (summary["entropy_H"] * summary["weight"]).sum()
print(f"\nOnset entropy tokens H: {weighted_H:.4f} bits")


Onset entropy tokens H: 0.0774 bits


#### *без частот -- для графем*

In [646]:
no_freq_gp_onset_df = gp_onset_df.copy()

In [647]:
# агрегируем
no_freq_gp_onset_df = (
    no_freq_gp_onset_df.groupby(["grapheme", "phoneme_real"], as_index=False)
      .agg({"word_count": "sum"})
)

def calc_no_freq_gp_onset_entropy(group):
    total = group["word_count"].sum()
    probs = group["word_count"] / total

    H = -np.sum(probs * np.log2(probs))

    return pd.Series({
        "entropy_H": H,
        "word_count_sum": total,
    })

summary = no_freq_gp_onset_df.groupby("grapheme").apply(calc_no_freq_gp_onset_entropy).reset_index()

# веса
total_all = summary["word_count_sum"].sum()
summary["weight"] = summary["word_count_sum"] / total_all

/tmp/ipykernel_3107/1065161001.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = no_freq_gp_onset_df.groupby("grapheme").apply(calc_no_freq_gp_onset_entropy).reset_index()


In [648]:
weighted_H = (summary["entropy_H"] * summary["weight"]).sum()
print(f"Onset entropy types: {weighted_H:.4f} bits")

Onset entropy types: 0.6483 bits


#### *без частот -- для фонем*

In [649]:
no_freq_pg_onset_df = pg_onset_df.copy()

In [650]:
# агрегируем
no_freq_pg_onset_df = (
    no_freq_pg_onset_df.groupby(["phoneme_real", "grapheme"], as_index=False)
      .agg({"word_count": "sum"})
)

summary = no_freq_gp_onset_df.groupby("phoneme_real").apply(calc_no_freq_gp_onset_entropy).reset_index()

# веса
total_all = summary["word_count_sum"].sum()
summary["weight"] = summary["word_count_sum"] / total_all

/tmp/ipykernel_3107/1864649964.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = no_freq_gp_onset_df.groupby("phoneme_real").apply(calc_no_freq_gp_onset_entropy).reset_index()


In [651]:
weighted_H = (summary["entropy_H"] * summary["weight"]).sum()
print(f"Onset entropy types: {weighted_H:.4f} bits")

Onset entropy types: 0.1053 bits


## *Consistency*


### Consitency: графемы - фонемы

In [658]:
try:
  gp_consistency = gp_entropy.copy()
except NameError:
  gp_consistency = pd.read_csv("gp_entropy.csv")

In [659]:
def compute_dominant_grapheme_consistency(df, label=""):
  """
  Сумма токенов наиболее частой фонемы для каждой графемы,
  делённая на общее количество пар графема–фонема в корпусе.
  """
  # общее количество пар графема–фонема
  total_pairs = df['total_freq_sum'].sum()

  # сумма частот доминирующей фонемы по каждой графеме
  dominant_sum = df['top_phoneme_freq'].sum()


  consistency = dominant_sum / total_pairs

  print(f"Total pairs         : {total_pairs:,.0f}")
  print(f"Dominant sum        : {dominant_sum:,.0f}")
  print(f"Consistency         : {consistency:.4f}")

In [660]:
gp_consistency_results = compute_dominant_grapheme_consistency(gp_consistency)

Total pairs         : 655,766
Dominant sum        : 481,068
Consistency         : 0.7336




---

### Consistency: фонемы - графемы

In [661]:
try:
  pg_consistency = pg_entropy_no_dash.copy()
except NameError:
  pg_consistency = pd.read_csv("pg_entropy_no_dash.csv")


In [662]:
def compute_dominant_grapheme_consistency(df, label=""):
  """
  Сумма токенов наиболее частой графемы для каждой фонемы,
  делённая на общее количество пар фонема-графема в корпусе.
  """
  # общее количество пар графема–фонема
  total_pairs = pg_consistency['total_freq_sum'].sum()

  # сумма частот доминирующей пары
  dominant_sum = pg_consistency['top_grapheme_freq'].sum()


  consistency = dominant_sum / total_pairs

  print(f"Total pairs         : {total_pairs:,.0f}")
  print(f"Dominant sum        : {dominant_sum:,.0f}")
  print(f"Consistency         : {consistency:.4f}")

In [663]:
pg_consistency_results = compute_dominant_grapheme_consistency(pg_consistency)

Total pairs         : 645,213
Dominant sum        : 519,665
Consistency         : 0.8054


## *Efficiency through Mutual Information*

In [664]:
try:
  efficiency_df = prob_pg.copy()
except NameError:
  efficiency_df = pd.read_csv("phoneme_grapheme_probabilities.csv")

In [665]:
def compute_mutual_information(df, label=""):
  """
  Вычисляет взаимную информацию I(G;P) между графемами и фонемами.

  Параметры:
    df    : DataFrame со столбцами 'phoneme_real', 'grapheme', 'total_freq'
    label : метка для идентификации результата

  Возвращает словарь:
    total_pairs      — общее число пар графема–фонема
    H_P              — энтропия фонем H(P), бит
    H_P_given_G      — условная энтропия H(P|G), бит
    I_GP             — взаимная информация I(G;P) = H(P) − H(P|G), бит
    efficiency       — доля энтропии фонем, объясняемая графемами: I(G;P) / H(P)
    grapheme_detail  — детальная разбивка энтропии по каждой графеме
  """
  # H(P): энтропия для фонемы
  phoneme_totals = df.groupby('phoneme_real')['total_freq'].sum()
  total_pairs    = phoneme_totals.sum()
  p_phoneme      = phoneme_totals / total_pairs
  H_P            = -(p_phoneme * np.log2(p_phoneme.clip(lower=1e-10))).sum()

  # H(P|G): условная энтропия (с учетом графемы)
  def grapheme_entropy(group):
    total = group['total_freq'].sum()
    p     = group['total_freq'] / total
    H     = -(p * np.log2(p.clip(lower=1e-10))).sum()
    return pd.Series({'grapheme_total': total, 'H_P_given_g': H})

  grapheme_summary          = df.groupby('grapheme').apply(grapheme_entropy).reset_index()
  grapheme_summary['p_g']   = grapheme_summary['grapheme_total'] / total_pairs
  grapheme_summary['wH']    = grapheme_summary['p_g'] * grapheme_summary['H_P_given_g']
  H_P_given_G               = grapheme_summary['wH'].sum()

  # Mutual information
  I_GP       = H_P - H_P_given_G

  # Efficiency
  efficiency = I_GP / H_P

  return {
    'label':       label,
    'total_pairs': total_pairs,
    'H_P':         H_P,
    'H_P_given_G': H_P_given_G,
    'I_GP':        I_GP,
    'efficiency':  efficiency,
    'grapheme_detail': grapheme_summary
  }

In [667]:
results_with_null = compute_mutual_information(
  efficiency_df,
  label="с учетом '-' (null phonemes included)"
)
results_without_null = compute_mutual_information(
  efficiency_df[efficiency_df['phoneme_real'] != '-'].copy(),
  label="без '-' (null phonemes excluded)"
)


comparison = pd.DataFrame([
  {k: v for k, v in r.items() if k != 'grapheme_detail'}
  for r in [results_with_null, results_without_null]
])
print(comparison)

                                   label  total_pairs       H_P  H_P_given_G  \
0  с учетом '-' (null phonemes included)       655766  4.802789     0.888738   
1       без '-' (null phonemes excluded)       634887  4.750508     0.864135   

       I_GP  efficiency  
0  3.914051    0.814954  
1  3.886373    0.818096  


/tmp/ipykernel_3107/2904322914.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grapheme_summary          = df.groupby('grapheme').apply(grapheme_entropy).reset_index()
/tmp/ipykernel_3107/2904322914.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grapheme_summary          = df.groupby('grapheme').apply(grapheme_entropy).reset_index()


## *Mean distance-based orthographic–phonological consistency (OPC)*

### получение соседей (дистанция = 1)

In [668]:
try:
  opc_df = cleaned_corpus.copy()
except NameError:
  opc_df = pd.read_csv("corr.csv")

In [669]:
ALPHABET = ru_alphabet

# 'word' > tuple
"""
при загрузке из файла может быть нужно lambda x: tuple(ast.literal_eval(x))
"""
# opc_df['word_tuple'] = opc_df['word'].apply(lambda x: tuple(ast.literal_eval(x)))
opc_df['word_tuple'] = opc_df['word'].apply(tuple)

# словарь для быстрого поиска
"""
при загрузке из файла может быть нужно lambda x: tuple(ast.literal_eval(x))
"""
# word_to_transcript = dict(zip(opc_df['word_tuple'], opc_df['transcript'].apply(lambda x: tuple(ast.literal_eval(x)))))
word_to_transcript = dict(zip(opc_df['word_tuple'], opc_df['transcript'].apply(tuple)))


# -------------- функции --------------

def gen_neighbours(word_tuple):
  """
  Генерирует слова с Levenshtein Distance == 1
  """
  neighbours = set()
  n = len(word_tuple)

  # substitution
  for i in range(n):
    original_char = word_tuple[i]
    for new_char in ALPHABET:
      if new_char != original_char:
        neighbours.add(word_tuple[:i] + (new_char,) + word_tuple[i+1:])

  # addition
  for i in range(n + 1):
    for new_char in ALPHABET:
      neighbours.add(word_tuple[:i] + (new_char,) + word_tuple[i:])

  # deletion
  if n > 1:
    for i in range(n):
      neighbours.add(word_tuple[:i] + word_tuple[i+1:])

  return neighbours

def get_neighbours(word_tuple):
  """
  возвращает транскрипции для след этапа
  """
  candidate_neighbours = gen_neighbours(word_tuple)
  existing_neighbours = candidate_neighbours.intersection(word_to_transcript.keys())

  return [list(word_to_transcript[nb]) for nb in existing_neighbours]

In [670]:
# применяем
opc_df['neighbours'] = opc_df['word_tuple'].apply(get_neighbours)
opc_df = opc_df.drop('word_tuple', axis=1)

In [671]:
# сохранить в файл
opc_df[["word", "transcript", "neighbours"]].to_csv("neighbours.csv")

In [672]:
# скачать
files.download("neighbours.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### подсчет дистанции транскрипций

In [673]:
try:
  df_neighbours = opc_df.copy()
except NameError:
  df_neighbours = pd.read_csv("neighbours.csv")
  df_neighbours = df_neighbours[df_neighbours['neighbours'].apply(lambda x: len(ast.literal_eval(x))) > 0].copy()

In [674]:
df_neighbours = df_neighbours[df_neighbours['neighbours'].apply(len) > 0].copy()
df_neighbours = df_neighbours.reset_index(drop=True)
df_neighbours = df_neighbours[["word", "transcript", "neighbours"]]

In [675]:
def avg_distance(row):
  transcript = row["transcript"]
  neighbours = row["neighbours"]

  distances = [distance(transcript, neib) for neib in neighbours]
  return sum(distances) / len(distances)

In [676]:
df_neighbours['distance'] = df_neighbours.apply(avg_distance, axis=1)

In [677]:
df_neighbours.head(3)

,word,transcript,neighbours,distance
0,"[а, в, г, у, с, т]","[a, v, g, u, s, t]","[[a, v, g, u, s', t', i]]",3.0
1,"[а, в, г, у, с, т, е]","[a, v, g, u, s', t', i]","[[a, v, g, u, s, t]]",3.0
2,"[а, в, т, о, м, а, т]","[a, f, t, a, m, a, t]","[[a, f, t, a, m, a, t, a]]",1.0


In [678]:
# ср арифметическое по выборке
mean_distance = df_neighbours['distance'].mean()

In [679]:
print(mean_distance)

1.4282658795259915
